## 方式1，使用with_structure_output函数

In [2]:
from langchain_openai import ChatOpenAI
from pydantic import Field, BaseModel
from rich import print as rprint

# 创建模型实例
model = ChatOpenAI(
    # model="mistral-nemo:latest",
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

class Movie(BaseModel):
    """电影详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分(10分制)")

s_model = model.with_structured_output(Movie,include_raw=True) # 使用了这个参数，输出格式会变，不再是class Movie而是dict
resp = s_model.invoke("给我介绍下电影《星际穿越》")
print(type(resp))
rprint(resp)



<class 'dict'>


{
    'raw': AIMessage(
        content='{"title": "星际穿越", "year": 2014, "director": "克里斯托弗·诺兰", "rating": 8.8}',
        additional_kwargs={
            'parsed': Movie(title='星际穿越', year=2014, director='克里斯托弗·诺兰', rating=8.8),
            'refusal': None
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 719,
                'prompt_tokens': 18,
                'total_tokens': 737,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
            },
            'model_provider': 'openai',
            'model_name': 'qwen3-vl:latest',
            'system_fingerprint': 'fp_ollama',
            'id': 'chatcmpl-209',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--01a0e901-ec6d-7623-9a76-16d51bc0fa41-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 18,
            'output_tokens': 719,
            'total_tokens': 737,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {}
        }
    ),
    'parsed': Movie(title='星际穿越', year=2014, director='克里斯托弗·诺兰', rating=8.8),
    'parsing_error': None
}

## 方式2： 使用输出解析器，不推荐

In [9]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import Field, BaseModel
from rich import print as rprint

# 1.创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
      ("system","回答优化问题，必须始终输出一个包含title(电影标题)和year(上映年份)的JSON对象"),
      ("human", "问题: {question} ")
])
# 2.创建模型实例
model = ChatOpenAI(
    # model="mistral-nemo:latest",
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)
# 3.定义结构
class Movie(BaseModel):
    """电影详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")

# 4.创建解析器
parser = JsonOutputParser(pydantic_object=Movie)

# 5.创建链
chain = prompt_template | model | parser
# 6.调用，返回字典
resp = chain.invoke({"question":"请介绍电影盗梦空间"})
print(resp)

{'title': '盗梦空间', 'year': 2010}
